# Cross-Domain Review Sentiment Classification

This notebook implements Option 2 (Review Sentiment Analysis) for ARTI 404. It trains a binary sentiment classifier (Positive/Negative) on Amazon product and IMDb movie reviews, then evaluates it on an in-domain held-out test and on a cross-domain Yelp restaurant test.

Experimental question: which method performs better on restaurant reviews when trained on product and movie reviews? The main judgment is Macro F1 on the Yelp test, with Accuracy reported alongside; the in-domain test is a supporting comparison only. Two methods are compared: TF-IDF + Logistic Regression (Method 1) and frozen MiniLM sentence embeddings + Logistic Regression (Method 2). Settings for both methods are selected on Development data only; neither test set is used for tuning.

## Setup

Import the libraries used across the notebook.

In [1]:
import platform
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import sklearn
import torch

c:\Users\moham\Desktop\nlp project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fix the random seed so splits, shuffling and model fitting are reproducible.

In [2]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Define relative project paths and create the output folders if they do not already exist. The notebook is expected to run from the project root, so every path here is relative rather than a path tied to one member's machine.

In [3]:
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
MODELS = PROJECT_ROOT / "models"

for path in (DATA_RAW, DATA_PROCESSED, RESULTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

Record the exact package versions used in this run, so the environment can be reproduced later.

In [4]:
print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)

Python: 3.12.10
numpy: 2.5.4
pandas: 3.1.1
scikit-learn: 1.9.1
sentence-transformers: 6.2.0
torch: 2.15.0+cpu


## Data

Download the UCI Sentiment Labelled Sentences archive into `data/raw/` if it is not already there, and extract it. The download is skipped when the three required files are already present; `__MACOSX` entries added by macOS zip tools are ignored everywhere below.

In [5]:
import datetime
import json
import urllib.request
import zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
RAW_ZIP_PATH = DATA_RAW / "sentiment+labelled+sentences.zip"
DOWNLOAD_INFO_PATH = DATA_RAW / "download_info.json"
REQUIRED_FILES = ["amazon_cells_labelled.txt", "imdb_labelled.txt", "yelp_labelled.txt"]


def find_file(filename):
    matches = [p for p in DATA_RAW.rglob(filename) if "__MACOSX" not in p.parts]
    return matches[0] if matches else None


existing_files = {name: find_file(name) for name in REQUIRED_FILES}

if all(existing_files.values()):
    print("Required data files already present in data/raw/, skipping download.")
    if DOWNLOAD_INFO_PATH.exists():
        recorded = json.loads(DOWNLOAD_INFO_PATH.read_text())
        print("Recorded download date:", recorded["download_date"])
    else:
        print("Recorded download date: unknown (no download_info.json found).")
else:
    download_date = datetime.date.today().isoformat()
    urllib.request.urlretrieve(DATA_URL, RAW_ZIP_PATH)
    with zipfile.ZipFile(RAW_ZIP_PATH) as archive:
        archive.extractall(DATA_RAW)
    DOWNLOAD_INFO_PATH.write_text(
        json.dumps({"source": DATA_URL, "download_date": download_date}, indent=2)
    )
    existing_files = {name: find_file(name) for name in REQUIRED_FILES}
    print(f"Downloaded and extracted on {download_date}.")

print("\nFiles found in data/raw/ (excluding __MACOSX):")
for path in sorted(DATA_RAW.rglob("*")):
    if path.is_file() and "__MACOSX" not in path.parts:
        print(" -", path.relative_to(PROJECT_ROOT))

missing = [name for name, path in existing_files.items() if path is None]
if missing:
    raise FileNotFoundError(f"Required file(s) not found after download: {missing}")

Required data files already present in data/raw/, skipping download.
Recorded download date: 2026-11-11

Files found in data/raw/ (excluding __MACOSX):
 - data\raw\download_info.json
 - data\raw\sentiment labelled sentences\.DS_Store
 - data\raw\sentiment labelled sentences\amazon_cells_labelled.txt
 - data\raw\sentiment labelled sentences\imdb_labelled.txt
 - data\raw\sentiment labelled sentences\readme.txt
 - data\raw\sentiment labelled sentences\yelp_labelled.txt
 - data\raw\sentiment+labelled+sentences.zip


Read each source file line by line and split on the last tab, since the label is always the final field and the review text itself may contain other whitespace. Lines that fail to parse are reported, not dropped silently. Cleaning is limited to Unicode normalisation and whitespace collapsing so negation words are preserved and the stored text is not lowercased.

In [6]:
import unicodedata


def clean_text(text: str) -> str:
    # NFKC normalises compatibility characters (e.g. full-width punctuation);
    # whitespace is then collapsed without touching negation words.
    normalized = unicodedata.normalize("NFKC", text)
    return " ".join(normalized.split())


def load_data(path: Path, source: str) -> tuple[pd.DataFrame, list[str]]:
    rows = []
    invalid_lines = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.rstrip("\n").rstrip("\r")
            if not line.strip():
                invalid_lines.append(f"{source} line {line_number}: empty line")
                continue
            if "\t" not in line:
                invalid_lines.append(f"{source} line {line_number}: no tab separator")
                continue
            text_part, _, label_part = line.rpartition("\t")
            text = clean_text(text_part)
            label_part = label_part.strip()
            if label_part not in ("0", "1"):
                invalid_lines.append(
                    f"{source} line {line_number}: invalid label {label_part!r}"
                )
                continue
            if not text:
                invalid_lines.append(f"{source} line {line_number}: empty text")
                continue
            rows.append(
                {
                    "example_id": f"{source}_{line_number:04d}",
                    "source": source,
                    "text": text,
                    "label": int(label_part),
                }
            )
    return pd.DataFrame(rows), invalid_lines

Load all three sources and collect any invalid lines found during parsing.

In [7]:
FILENAME_BY_SOURCE = {
    "amazon": "amazon_cells_labelled.txt",
    "imdb": "imdb_labelled.txt",
    "yelp": "yelp_labelled.txt",
}

frames = []
all_invalid_lines = []
for source, filename in FILENAME_BY_SOURCE.items():
    frame, invalid_lines = load_data(existing_files[filename], source)
    frames.append(frame)
    all_invalid_lines.extend(invalid_lines)

raw_df = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(raw_df)} valid rows from {len(FILENAME_BY_SOURCE)} source files.")
print(f"Invalid lines found: {len(all_invalid_lines)}")
for entry in all_invalid_lines:
    print(" -", entry)

Loaded 3000 valid rows from 3 source files.
Invalid lines found: 0


Counts by source and label, before any duplicate handling.

In [8]:
count_table = (
    raw_df.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
count_table["total"] = count_table["negative_0"] + count_table["positive_1"]
count_table

label,negative_0,positive_1,total
source,,,
amazon,500,500,1000
imdb,500,500,1000
yelp,500,500,1000


Build a `duplicate_key` for auditing only (lowercased, whitespace-collapsed text; punctuation and negation words are kept). The stored `text` column is never changed by this key.

In [9]:
def make_duplicate_key(text: str) -> str:
    return " ".join(text.lower().split())


raw_df["duplicate_key"] = raw_df["text"].apply(make_duplicate_key)

Audit duplicate_key groups across all three sources. A group with conflicting labels is dropped entirely and logged as a conflict. A group sharing one label keeps exactly one row, chosen by a fixed source priority (Yelp, then Amazon, then IMDb, then order of appearance in the source file) so a sentence that occurs in Yelp stays in the cross-domain test source rather than in training. No model output is used in this decision.

In [10]:
SOURCE_PRIORITY = {"yelp": 0, "amazon": 1, "imdb": 2}


def audit_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    kept_rows = []
    removed_records = []
    within_source_keys = []
    across_source_keys = []
    conflict_keys = []

    for key, group in df.groupby("duplicate_key", sort=False):
        if len(group) == 1:
            kept_rows.append(group.iloc[0])
            continue

        is_cross_source = group["source"].nunique() > 1

        if group["label"].nunique() > 1:
            conflict_keys.append(key)
            for _, row in group.iterrows():
                removed_records.append(
                    {
                        "example_id": row["example_id"],
                        "source": row["source"],
                        "label": row["label"],
                        "duplicate_key": key,
                        "reason": "conflict",
                    }
                )
            continue

        if is_cross_source:
            across_source_keys.append(key)
        else:
            within_source_keys.append(key)

        ordered_group = group.assign(
            _priority=group["source"].map(SOURCE_PRIORITY)
        ).sort_values(["_priority", "example_id"])
        keep_row = ordered_group.iloc[0].drop(labels="_priority")
        drop_group = ordered_group.iloc[1:]

        kept_rows.append(keep_row)
        for _, row in drop_group.iterrows():
            removed_records.append(
                {
                    "example_id": row["example_id"],
                    "source": row["source"],
                    "label": row["label"],
                    "duplicate_key": key,
                    "reason": "duplicate",
                }
            )

    clean_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    removal_log = pd.DataFrame(
        removed_records, columns=["example_id", "source", "label", "duplicate_key", "reason"]
    )

    report = {
        "duplicate_groups_within_source": len(within_source_keys),
        "duplicate_groups_across_source": len(across_source_keys),
        "conflicting_groups": len(conflict_keys),
        "within_source_keys": within_source_keys,
        "across_source_keys": across_source_keys,
        "conflict_keys": conflict_keys,
    }

    return clean_df, removal_log, report

Run the audit and report the group and row counts.

In [11]:
clean_df, removal_log, duplicate_report = audit_duplicates(raw_df)

rows_removed_as_duplicate = int((removal_log["reason"] == "duplicate").sum())
rows_removed_as_conflict = int((removal_log["reason"] == "conflict").sum())

print("Duplicate groups within a single source:", duplicate_report["duplicate_groups_within_source"])
print("Duplicate groups across sources:", duplicate_report["duplicate_groups_across_source"])
print("Conflicting-label groups (dropped entirely):", duplicate_report["conflicting_groups"])
print("Rows removed as duplicates:", rows_removed_as_duplicate)
print("Rows removed as conflicts:", rows_removed_as_conflict)
print("Rows remaining:", len(clean_df))

Duplicate groups within a single source: 18
Duplicate groups across sources: 1
Conflicting-label groups (dropped entirely): 0
Rows removed as duplicates: 21
Rows removed as conflicts: 0
Rows remaining: 2979


Show a few example duplicate-key groups, and a conflicting-label group if one exists, so the kind of overlap being removed is visible.

In [12]:
example_duplicate_keys = (
    duplicate_report["within_source_keys"] + duplicate_report["across_source_keys"]
)[:5]

print("Example duplicate-key groups (same label, one row kept):")
for key in example_duplicate_keys:
    group = raw_df[raw_df["duplicate_key"] == key]
    print(f"\nduplicate_key: {key!r}")
    print(group[["example_id", "source", "label", "text"]].to_string(index=False))

if duplicate_report["conflict_keys"]:
    print("\n\nExample conflicting-label group (whole group dropped):")
    key = duplicate_report["conflict_keys"][0]
    group = raw_df[raw_df["duplicate_key"] == key]
    print(f"duplicate_key: {key!r}")
    print(group[["example_id", "source", "label", "text"]].to_string(index=False))
else:
    print("\n\nNo conflicting-label groups were found.")

Example duplicate-key groups (same label, one row kept):

duplicate_key: 'works great!.'
 example_id source  label          text
amazon_0019 amazon      1 Works great!.
amazon_0525 amazon      1 Works great!.

duplicate_key: 'very disappointed.'
 example_id source  label               text
amazon_0057 amazon      0 VERY DISAPPOINTED.
amazon_0144 amazon      0 very disappointed.

duplicate_key: 'if you like a loud buzzing to override all your conversations, then this phone is for you!'
 example_id source  label                                                                                       text
amazon_0180 amazon      0 If you like a loud buzzing to override all your conversations, then this phone is for you!
amazon_0745 amazon      0 If you like a loud buzzing to override all your conversations, then this phone is for you!

duplicate_key: "don't buy this product."
 example_id source  label                    text
amazon_0181 amazon      0 Don't buy this product.
amazon_0544 amazo

Save the per-source, per-label audit trail: original count, rows removed as duplicates, rows removed as conflicts, and the final count.

In [13]:
original_counts = raw_df.groupby(["source", "label"]).size().rename("original_count")

removed_duplicate_counts = (
    removal_log[removal_log["reason"] == "duplicate"]
    .groupby(["source", "label"])
    .size()
    .rename("removed_as_duplicate")
)
removed_conflict_counts = (
    removal_log[removal_log["reason"] == "conflict"]
    .groupby(["source", "label"])
    .size()
    .rename("removed_as_conflict")
)
final_counts = clean_df.groupby(["source", "label"]).size().rename("final_count")

data_audit = (
    pd.concat(
        [original_counts, removed_duplicate_counts, removed_conflict_counts, final_counts],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

data_audit.to_csv(RESULTS / "data_audit.csv", index=False)
data_audit

,source,label,original_count,removed_as_duplicate,removed_as_conflict,final_count
0,amazon,0,500,4,0,496
1,amazon,1,500,9,0,491
2,imdb,0,500,2,0,498
3,imdb,1,500,2,0,498
4,yelp,0,500,3,0,497
5,yelp,1,500,1,0,499


Counts by source and label after cleaning.

In [14]:
final_count_table = (
    clean_df.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
final_count_table["total"] = final_count_table["negative_0"] + final_count_table["positive_1"]
final_count_table

label,negative_0,positive_1,total
source,,,
amazon,496,491,987
imdb,498,498,996
yelp,497,499,996


Limitation: the UCI source does not provide the original review ID for each sentence, so sentences drawn from the same original review cannot be guaranteed to stay in the same split. Only exact duplicates, matched after case and whitespace normalisation, are detected and handled above; near-duplicate or paraphrased sentences are not checked automatically.